# Dedicated model for translation

## MADLAD

In [ ]:
%pip install transformers torch==2.5 accelerate bitsandbytes>=0.46.1

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, BitsAndBytesConfig
import torch

In [ ]:
# quantization_config = BitsAndBytesConfig(
#     # load_in_8bit = True
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_quant_type="nf4",            # Better accuracy than standard fp4
#     # bnb_4bit_use_double_quant=True        # Compresses the quantization constants
# )

In [ ]:
model_name = 'google/madlad400-10b-mt'
model = T5ForConditionalGeneration.from_pretrained(
    model_name, 
    device_map="cuda", 
    torch_dtype=torch.float32,
    # quantization_config=quantization_config
)
tokenizer = T5Tokenizer.from_pretrained(model_name)

In [ ]:
# text = "<2id> What a beautiful day it is today. I am so happy to be alive and enjoy the sunshine. <num> min of pure bliss."
text = "<2id> i love waiting <num> min for a cab - such shortage ....... <user> please allow uber . this is insane ."
input_ids = tokenizer(text, return_tensors="pt").input_ids.to('cuda')
outputs = model.generate(
    input_ids=input_ids,
    # do_sample=False,            # Forces 100% deterministic, repeatable outputs
    # num_beams=4,                # Explores 4 parallel translation paths for highest quality
    # length_penalty=0.6,         # Prevents the model from being overly wordy or cutting off
    # no_repeat_ngram_size=3,     # Blocks repetitive loops
    # repetition_penalty=1.2,     # Discourages repeating the source text
    # max_new_tokens=512,         # Ensures long paragraphs aren't cut short
    # early_stopping=True         # Stops instantly when the translation is complete
)


result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

## NLBB

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch

In [ ]:
# quantization_config = BitsAndBytesConfig(
#     load_in_8bit = True
#     # load_in_4bit=True,
#     # bnb_4bit_compute_dtype=torch.float16,
#     # bnb_4bit_quant_type="nf4",            # Better accuracy than standard fp4
#     # bnb_4bit_use_double_quant=True        # Compresses the quantization constants
# )

In [ ]:
# Load tokenizer and model directly
model_name = "facebook/nllb-200-3.3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    # quantization_config=quantization_config, 
    dtype=torch.float16,
    device_map="auto"
    )

In [ ]:
# Prepare text and set language tokens (e.g., English to French)
# text = "What a successful toast, it looks so delicious!"
tokenizer.src_lang = "eng_Latn"
inputs = tokenizer(text, return_tensors="pt").to("cuda")

# Generate translation using target language code token (e.g., french: fra_Latn)
translated_tokens = model.generate(
    **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("ind_Latn")
)
output = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
print(output)